In [1]:
from env import SimpleARGEnvironment
from utils import load_sequences

In [3]:
Ne = 10000
r_per_bp = 2e-8
mu_per_bp = 2e-8

dataset_path="validation/fasta/sim_l25kb_0.fa"

sequences = load_sequences(dataset_path)

In [4]:
env = SimpleARGEnvironment(
    num_sequences=len(sequences),
    population_size=Ne,
    recombination_rate=r_per_bp,
    mutation_rate=mu_per_bp,
    sequences=sequences,
    seed=7,
    bp_per_blocks=1
)

episodes = 1

In [5]:
from tb_gfn import TBGFlowNetGenerator
from rollout_worker_arg import RolloutWorker

generator = TBGFlowNetGenerator(env, 1)
model = generator.arg_model
rollout_worker = RolloutWorker(env)

In [6]:
ret, trajectories = rollout_worker.rollout(generator, episodes=1)

batch_active_lineage_counts tensor([8])
batch_active_lineage_counts tensor([9])
batch_active_lineage_counts tensor([10])
batch_active_lineage_counts tensor([11])
batch_active_lineage_counts tensor([10])
batch_active_lineage_counts tensor([9])
batch_active_lineage_counts tensor([10])
batch_active_lineage_counts tensor([11])
batch_active_lineage_counts tensor([12])
batch_active_lineage_counts tensor([13])
batch_active_lineage_counts tensor([12])
batch_active_lineage_counts tensor([13])
batch_active_lineage_counts tensor([12])
batch_active_lineage_counts tensor([11])
batch_active_lineage_counts tensor([10])
batch_active_lineage_counts tensor([11])
batch_active_lineage_counts tensor([10])
batch_active_lineage_counts tensor([9])
batch_active_lineage_counts tensor([8])
batch_active_lineage_counts tensor([9])
batch_active_lineage_counts tensor([10])
batch_active_lineage_counts tensor([11])
batch_active_lineage_counts tensor([10])
batch_active_lineage_counts tensor([11])
batch_active_lineage_c

In [8]:
t = 0
s = trajectories[0].transitions[t][0]
s_next = trajectories[0].transitions[t][1]
a = trajectories[0].actions[t]

In [16]:
import math 
log_pf = ret["log_paths_pf"][0, t].item()
num_parents = generator.count_backward_parents(s_next)
log_pb = -math.log(num_parents)

print("action:", a)
print("PF:", math.exp(log_pf))
print("PB:", math.exp(log_pb))
print("logPF:", log_pf)
print("logPB:", log_pb)
print("logPF - logPB:", log_pf - log_pb)

action: RecombinationChoice(active_lineage_i=2, material_count=25000, span_start=0, span_end=24999, time_action=9, breakpoint=6409)
PF: 1.776014112737704e-07
PB: 1.0
logPF: -15.543724060058594
logPB: -0.0
logPF - logPB: -15.543724060058594


In [17]:
log_pf = ret["log_paths_pf"][0].sum()
log_pb = ret["log_paths_pb"][0].sum()
log_r = ret["log_rewards"][0]
log_z = generator.compute_log_Z().detach()

residual = log_z + log_pf - (log_r + log_pb)
print(residual.item())

36358.92578125


In [19]:
log_r

tensor(-37294.4180)

In [ ]:
from env import Trajectory
states = [env.get_initial_state() for _ in range(episodes)]
trajectories = [Trajectory(x) for x in states]

unfinished = [idx for idx, state in enumerate(states) if not state.is_done]
active_states = [states[idx] for idx in unfinished]


input_dict = env.prepare_state_rollout_inputs(
    active_states,
    random_spec=None
    )

In [ ]:
## Encode states

states = input_dict["states"]
active_counts = [len(state.active_lineages) for state in states]

lineage = states[0].active_lineages[0]
feature = lineage.partials

In [ ]:
weights = model._material_segments_masking(lineage.material_segments, device=env.device, dtype=env.seq_arrays.dtype)

In [ ]:
masked_feature = feature * weights[:, None]

In [ ]:
normalize_feature = env.evolution_model.normalize_partials(masked_feature)

In [ ]:
normalize_feature